# HargaIkan v2 · 01 Ingest

Thin Colab wrapper around `hargaikan.pipeline.ingest` (the same code as `scripts/ingest.py`).
It cleans the IkanSight snapshot CSVs, builds price bands and publishes them to Supabase through
stage → validate → promote. This notebook has no logic of its own.

Secrets (Colab → Secrets panel): `APP_ENV`, `SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY`.
The service role key is used only here and in `scripts/*.py`, never in the app. Values are never printed.

In [ ]:
# 1. Get the code and install the package (replace the placeholder URL with your repository).
REPO_URL = "https://github.com/YOUR_GITHUB_USER/HargaIkan.git"
!git clone -q {REPO_URL} HargaIkan || true
%cd HargaIkan
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
# 2. Copy secrets from Colab userdata into the environment (names only are printed).
import os
from google.colab import userdata

for name in ("APP_ENV", "SUPABASE_URL", "SUPABASE_SERVICE_ROLE_KEY"):
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: set")
    except Exception:
        print(f"{name}: missing")

In [ ]:
# 3. Upload the snapshot CSVs (ikansight_price_per_kg_by_location*.csv, optional ikansight_fish_coverage.csv).
#    Skip this cell for a --synthetic run without raw files.
import shutil
from pathlib import Path
from google.colab import files

Path("data/raw").mkdir(parents=True, exist_ok=True)
for filename in files.upload():
    shutil.move(filename, Path("data/raw") / filename)
print(sorted(p.name for p in Path("data/raw").glob("*.csv")))

In [ ]:
# 4. Parameters: the same flags as scripts/ingest.py. TARGET must equal APP_ENV.
ARGS = ["--raw", "data/raw", "--target", "development", "--dry-run"]
# Examples:
#   ARGS = ["--raw", "data/raw", "--target", "development"]                 # publish real data
#   ARGS = ["--synthetic", "--target", "development", "--dry-run"]          # demo data, no writes

In [ ]:
# 5. Run the shared pipeline. Exit code: 0 ok, 1 failure, 2 usage/configuration error.
from hargaikan.pipeline.ingest import main

exit_code = main(ARGS)
print("exit code:", exit_code)